In [1]:
# Instala condacolab, que permite tener un entorno conda/mamba real en Colab
!pip install -q condacolab
import condacolab
condacolab.install()   # Esto reinicia el kernel de Colab automáticamente
#asegurate de volver a correr hasta que apareza Everything looks OK!

✨🍰✨ Everything looks OK!


In [2]:
# Revisa qué versión de Python quedó "fijada" (pin) por condacolab
!cat /usr/local/conda-meta/pinned

python 3.12.*
python_abi 3.12.* *cp312*
cudatoolkit *.*.*


In [3]:
# Elimina el pin conflictivo: así mamba puede resolver libremente
# las dependencias de sage sin chocar contra una versión de Python fija
!rm -f /usr/local/conda-meta/pinned

In [4]:
# Instala SageMath completo vía mamba (tarda varios minutos, es normal)
!mamba install -y -c conda-forge sage


Looking for: ['sage']

[+] 0.0s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  [+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   3%
conda-forge/noarch    18%[+] 0.3s
conda-forge/linux-64   9%
conda-forge/noarch    30%[+] 0.4s
conda-forge/linux-64  14%
conda-forge/noarch    40%[+] 0.5s
conda-forge/linux-64  19%
conda-forge/noarch    50%[+] 0.6s
conda-forge/linux-64  22%
conda-forge/noarch    57%[+] 0.7s
conda-forge/linux-64  24%
conda-forge/noarch    61%[+] 0.8s
conda-forge/linux-64  27%
conda-forge/noarch    66%[+] 0.9s
conda-forge/linux-64  29%
conda-forge/noarch    70%[+] 1.0s
conda-forge/linux-64  29%
conda-forge/noarch    70%[+] 1.1s
conda-forge/linux-64  29%
conda-forge/noarch    70%[+] 1.2s
conda-forge/linux-64  29%
conda-forge/noarch    70%[+] 1.3s
conda-forge/linux-64  29%
conda-forge/noarch    70%[+] 1.4s
conda-forge/linux-64  29%
conda-forge/noarch    70%[+] 1.5s
conda-forge/linux-64  29%
conda-forge/noarch    71%[+] 1.6s
conda-f

In [20]:
import subprocess, tempfile, os, json

def ejecutar_sage(codigo_sage: str, timeout: int = 1800) -> str:
    """
    Ejecuta código Sage como proceso externo (binario 'sage'), evitando
    el import directo dentro del kernel de Colab (incompatible por
    versión de Python). Devuelve la salida estándar como texto.
    """
    with tempfile.NamedTemporaryFile(mode='w', suffix='.sage', delete=False) as f:
        f.write(codigo_sage)
        ruta = f.name

    try:
        resultado = subprocess.run(
            ["sage", ruta], capture_output=True, text=True, timeout=timeout
        )
    finally:
        os.unlink(ruta)

    if resultado.returncode != 0:
        raise RuntimeError(f"Error ejecutando Sage:\n{resultado.stderr}")

    return resultado.stdout

In [21]:
_PLANTILLA_SAGE_BATCH = r'''
from sage.all import *
import json

var_name = "%%VAR%%"
polinomios = json.loads(r"""%%POLYS_JSON%%""")
R = PolynomialRing(QQ, var_name)

resultados = []
for poly_str in polinomios:
    entry = {"polinomio": poly_str}
    try:
        f = R(poly_str)
        if not f.is_irreducible():
            raise ValueError("El polinomio no es irreducible.")

        n = f.degree()
        K = NumberField(f, 'a')
        disc = K.discriminant()
        sig = K.signature()
        ramified = sorted(disc.prime_factors())

        G = f.galois_group()
        grupo_perm = G.group() if hasattr(G, "group") else G
        Gfresh = PermutationGroup(grupo_perm.gens(), canonicalize=False)

        grado_perm = Gfresh.degree()
        orden_galois = Gfresh.order()
        # libgap (enlace C directo) evita el bug de parsing de la consola de texto 'gap.xxx()'
        t_num = Integer(libgap.TransitiveIdentification(Gfresh.gap()))
        label = "%sT%s" % (grado_perm, t_num)

        try:
            nombre = str(Gfresh.structure_description())
        except Exception:
            nombre = label

        es_abeliano = bool(Gfresh.is_abelian())
        es_ciclico = bool(Gfresh.is_cyclic())
        es_galois = bool(K.is_galois())

        # float(...) exterior obligatorio: RealNumber de Sage no es serializable a JSON
        root_disc = float(float(abs(disc)) ** (1.0 / n))

        if es_galois:
            galois_root_disc = root_disc
            galois_root_disc_exacto = None
            if len(ramified) == 1:
                p = ramified[0]
                e = disc.valuation(p)
                galois_root_disc_exacto = "%s^(%s/%s)" % (p, e, n)
        else:
            galois_root_disc = None
            galois_root_disc_exacto = None
            if orden_galois <= 48:
                try:
                    L = K.galois_closure('b')
                    discL = L.discriminant()
                    galois_root_disc = float(float(abs(discL)) ** (1.0 / orden_galois))
                except Exception:
                    pass

        d_libre = disc.squarefree_part()
        disc_root_field = "Q" if d_libre == 1 else "Q(sqrt(%s))" % d_libre

        aut_orden = len(K.automorphisms())
        num_clases = len(Gfresh.conjugacy_classes())

        try:
            h = int(K.class_number(proof=False))
        except Exception:
            h = None

        entry.update({
            "ok": True,
            "grado": int(n),
            "signatura": [int(sig[0]), int(sig[1])],
            "discriminante": str(disc),
            "discriminante_factorizado": str(disc.factor()),
            "primos_ramificados": [str(p) for p in ramified],
            "root_discriminante": root_disc,
            "galois_root_discriminante": galois_root_disc,
            "galois_root_discriminante_exacto": galois_root_disc_exacto,
            "discriminante_root_field": disc_root_field,
            "grupo_orden": int(orden_galois),
            "grupo_label": str(label),
            "grupo_nombre": nombre,
            "es_galois": es_galois,
            "es_abeliano": es_abeliano,
            "es_ciclico": es_ciclico,
            "aut_orden": int(aut_orden),
            "num_clases_conjugacion": int(num_clases),
            "numero_clase": h,
        })
    except Exception as e:
        entry.update({"ok": False, "error": str(e)})

    resultados.append(entry)

print("===JSON_START===")
print(json.dumps(resultados))
print("===JSON_END===")
'''

In [22]:
def analizar_polinomios(lista_polys: list, var_name: str = 'x', timeout: int = 1800) -> list:
    """Analiza N polinomios en UN solo proceso Sage. Devuelve lista de dicts."""
    polys_json = json.dumps(lista_polys)
    codigo = (_PLANTILLA_SAGE_BATCH
              .replace("%%VAR%%", var_name)
              .replace("%%POLYS_JSON%%", polys_json))

    salida = ejecutar_sage(codigo, timeout=timeout)
    inicio = salida.find("===JSON_START===") + len("===JSON_START===")
    fin = salida.find("===JSON_END===")
    return json.loads(salida[inicio:fin].strip())


def analizar_polinomio(poly_str: str, var_name: str = 'x') -> dict:
    """Analiza UN solo polinomio (reutiliza analizar_polinomios con lista de tamaño 1)."""
    d = analizar_polinomios([poly_str], var_name)[0]
    if not d.get("ok"):
        raise ValueError(f"Error en Sage: {d.get('error')}")
    return d

In [23]:
def _descripcion_grupo(d: dict) -> str:
    orden = d['grupo_orden']
    if d['es_ciclico']:
        return f"A cyclic group of order {orden}"
    if d['es_abeliano']:
        return f"An abelian group of order {orden}"
    return f"A group of order {orden}, isomorphic to {d['grupo_nombre']}"


def _imprimir_reporte(d: dict, poly_str: str):
    print("Invariants")
    print("-" * 60)
    print(f"Polynomial:                {poly_str}")
    print(f"Degree:                    {d['grado']}")
    print(f"Signature:                 ({d['signatura'][0]}, {d['signatura'][1]})")
    print(f"Discriminant:              {d['discriminante']} = {d['discriminante_factorizado']}")
    print(f"Root discriminant:         {d['root_discriminante']:.2f}")

    if d['galois_root_discriminante'] is not None:
        if d['galois_root_discriminante_exacto']:
            print(f"Galois root discriminant:  {d['galois_root_discriminante_exacto']} "
                  f"≈ {d['galois_root_discriminante']:.14f}")
        else:
            print(f"Galois root discriminant:  ≈ {d['galois_root_discriminante']:.14f}")
    else:
        print("Galois root discriminant:  (omitido: grupo demasiado grande)")

    print(f"Ramified primes:           {', '.join(d['primos_ramificados'])}")
    print(f"Discriminant root field:   {d['discriminante_root_field']}")

    if d['es_galois']:
        print(f"Aut(K/Q) = Gal(K/Q):       {d['grupo_nombre']}")
    else:
        print(f"Aut(K/Q):                  order {d['aut_orden']}")
        print(f"Gal(K/Q) [cierre normal]:  {d['grupo_nombre']} ({d['grupo_label']})")

    if d['es_galois'] and d['es_abeliano']:
        print("\nThis field is Galois and abelian over Q.")
    elif d['es_galois']:
        print("\nThis field is Galois over Q, but the Galois group is not abelian.")
    else:
        print("\nThis field is not Galois over Q.")

    print(f"Class number:              {d['numero_clase']}")
    print()
    print("Galois group")
    print("-" * 60)
    print(f"{d['grupo_nombre']} (as {d['grupo_label']}):")
    print()
    print(_descripcion_grupo(d))
    print(f"The {d['num_clases_conjugacion']} conjugacy class representatives for {d['grupo_nombre']}")


def mostrar_reporte(poly_str: str, var_name: str = 'x') -> dict:
    """Modo INDIVIDUAL: analiza e imprime un solo polinomio."""
    d = analizar_polinomio(poly_str, var_name)
    _imprimir_reporte(d, poly_str)
    return d


def mostrar_reportes_multiples(lista_polys: list, var_name: str = 'x') -> list:
    """Modo LOTE: analiza e imprime varios polinomios en un solo proceso Sage."""
    resultados = analizar_polinomios(lista_polys, var_name)
    for d in resultados:
        print("=" * 60)
        if d.get("ok"):
            _imprimir_reporte(d, d["polinomio"])
        else:
            print(f"Polynomial: {d['polinomio']}")
            print(f"ERROR: {d['error']}")
    print("=" * 60)
    return resultados

In [27]:
def consultar():
    """
    Menú principal: permite elegir entre analizar UN polinomio
    o VARIOS polinomios (separados por punto y coma ';').
    """
    print("¿Cuántos polinomios quieres analizar?")
    print("  1) Uno solo")
    print("  2) Varios (lote)")
    modo = input("Elige 1 o 2: ").strip()

    if modo == "1":
        entrada = input("Ingresa el polinomio en variable x (ej: x^5 - 2): ").strip()
        if not entrada:
            print("No ingresaste nada.")
            return
        try:
            mostrar_reporte(entrada)
        except Exception as e:
            print("Error:", e)

    elif modo == "2":
        print("Ingresa los polinomios separados por ';' (ej: x^5 - 2; x^3 - 2; x^4 + 1)")
        entrada = input("> ").strip()
        if not entrada:
            print("No ingresaste nada.")
            return
        lista = [p.strip() for p in entrada.split(";") if p.strip()]
        try:
            mostrar_reportes_multiples(lista)
        except Exception as e:
            print("Error:", e)

    else:
        print("Opción no válida. Elige 1 o 2.")

consultar()

¿Cuántos polinomios quieres analizar?
  1) Uno solo
  2) Varios (lote)
Elige 1 o 2: 1
Ingresa el polinomio en variable x (ej: x^5 - 2): x^5 - 2
Invariants
------------------------------------------------------------
Polynomial:                x^5 - 2
Degree:                    5
Signature:                 (1, 2)
Discriminant:              50000 = 2^4 * 5^5
Root discriminant:         8.71
Galois root discriminant:  ≈ 11.08254495193135
Ramified primes:           2, 5
Discriminant root field:   Q(sqrt(5))
Aut(K/Q):                  order 1
Gal(K/Q) [cierre normal]:  C5 : C4 (5T3)

This field is not Galois over Q.
Class number:              1

Galois group
------------------------------------------------------------
C5 : C4 (as 5T3):

A group of order 20, isomorphic to C5 : C4
The 5 conjugacy class representatives for C5 : C4


BLOQUES DE PRUEBA

In [ ]:
# Ejemplo A: un solo polinomio
resultado_unico = mostrar_reporte(
    "x^9 - x^8 - 8*x^7 + 7*x^6 + 21*x^5 - 15*x^4 - 20*x^3 + 10*x^2 + 5*x - 1"
)

In [25]:
# Ejemplo B: varios polinomios en un solo proceso Sage
resultados_lote = mostrar_reportes_multiples([
    "x^9 - x^8 - 8*x^7 + 7*x^6 + 21*x^5 - 15*x^4 - 20*x^3 + 10*x^2 + 5*x - 1",  # 9T1, C9
    "x^5 - 2",   # esperado: no abeliano
    "x^4 + 1",   # ciclotómico
    "x^3 - 2",   # S3, 3T2
])

Invariants
------------------------------------------------------------
Polynomial:                x^9 - x^8 - 8*x^7 + 7*x^6 + 21*x^5 - 15*x^4 - 20*x^3 + 10*x^2 + 5*x - 1
Degree:                    9
Signature:                 (9, 0)
Discriminant:              16983563041 = 19^8
Root discriminant:         13.70
Galois root discriminant:  19^(8/9) ≈ 13.69840075203899
Ramified primes:           19
Discriminant root field:   Q
Aut(K/Q) = Gal(K/Q):       C9

This field is Galois and abelian over Q.
Class number:              1

Galois group
------------------------------------------------------------
C9 (as 9T1):

A cyclic group of order 9
The 9 conjugacy class representatives for C9
Invariants
------------------------------------------------------------
Polynomial:                x^5 - 2
Degree:                    5
Signature:                 (1, 2)
Discriminant:              50000 = 2^4 * 5^5
Root discriminant:         8.71
Galois root discriminant:  ≈ 11.08254495193135
Ramified primes: 

In [28]:
# Reemplaza por el polinomio que te está fallando
poly_prueba = "x^9 + x^8 + x^6 + x^3"

resultado_crudo = analizar_polinomios([poly_prueba])
print(json.dumps(resultado_crudo, indent=2))

[
  {
    "polinomio": "x^9 + x^8 + x^6 + x^3",
    "ok": false,
    "error": "El polinomio no es irreducible."
  }
]


In [30]:
poly_prueba = "x^9 + x^8 + x^6 + x^3"   # irreducible por criterio de Eisenstein en p=2

resultado_crudo = analizar_polinomios([poly_prueba])
print(json.dumps(resultado_crudo, indent=2))

[
  {
    "polinomio": "x^9 + x^8 + x^6 + x^3",
    "ok": false,
    "error": "El polinomio no es irreducible."
  }
]
